# 03. Production growth and yield gaps

**Purpose:** find out whether Nigeria's production growth came from using more land or
from better yields, then compare its yields with other countries.

**Input:** `data/processed/crop_panel.csv`

**Outputs:** `decomposition.csv`, `yield_gap.csv`, `headline.json`

In [1]:
import sys, pathlib, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import pandas as pd
import numpy as np

from src import config as C

pd.set_option("display.width", 170)

panel = pd.read_csv(C.PROCESSED / "crop_panel.csv")
print("panel:", panel.shape)

panel: (6591, 6)


## 1. Pick the years

Yields move around a lot from one year to the next because of rainfall. Nigeria's yam
yield was 10.68, then 10.35, then 11.35 tonnes per hectare in 1990, 1991 and 1992.

So instead of one year at each end, I use the average of three.

In [2]:
window = panel[
    (panel["country"] == "Nigeria")
    & panel["year"].between(C.START_YEAR, C.END_YEAR)
].sort_values(["crop", "year"])

expected = len(C.CROPS) * (C.END_YEAR - C.START_YEAR + 1)

print("rows found:   ", len(window))
print("rows expected:", expected)

if len(window) != expected:
    raise ValueError("a crop is missing some years")

rows found:    272
rows expected: 272


The row count has to be exact. The next step takes the first three and last three
rows of each crop, which only gives the right years if no year is missing.

In [3]:
def endpoints(df, n_years):
    first = df.groupby("crop").head(n_years)
    last  = df.groupby("crop").tail(n_years)

    cols  = ["area_ha", "yield_t_ha"]
    start = first.groupby("crop")[cols].mean()
    end   = last.groupby("crop")[cols].mean()

    out = start.join(end, lsuffix="_start", rsuffix="_end").reset_index()
    out["prod_start"] = out["area_ha_start"] * out["yield_t_ha_start"]
    out["prod_end"]   = out["area_ha_end"] * out["yield_t_ha_end"]
    return out


ends = endpoints(window, C.ENDPOINT_WINDOW)
ends[["crop", "area_ha_start", "area_ha_end",
      "yield_t_ha_start", "yield_t_ha_end"]].round(2)

,crop,area_ha_start,area_ha_end,yield_t_ha_start,yield_t_ha_end
0,Cassava,2313376.67,10050238.67,10.81,6.05
1,Cowpea,2152580.00,4990290.33,0.66,0.85
2,Groundnut,960000.00,3517403.33,1.37,1.22
3,Maize,5156333.33,5901760.00,1.13,2.08
4,Millet,4568333.33,1922463.33,1.00,0.95
5,Rice,1508000.00,4469966.67,1.99,1.92
6,Sorghum,5065666.67,5776966.67,1.02,1.15
7,Yam,1552666.67,7526194.33,10.79,8.12


### Does it matter which years I pick?

I ran the same sum nine times, using every pairing of a possible start year with a
possible end year.

In [4]:
single = []

for sy in range(C.START_YEAR, C.START_YEAR + C.ENDPOINT_WINDOW):
    for ey in range(C.END_YEAR - C.ENDPOINT_WINDOW + 1, C.END_YEAR + 1):
        start = window[window["year"] == sy].set_index("crop")
        end   = window[window["year"] == ey].set_index("crop")
        crops = start.index.intersection(end.index)

        a0, y0 = start.loc[crops, "area_ha"], start.loc[crops, "yield_t_ha"]
        a1, y1 = end.loc[crops, "area_ha"],   end.loc[crops, "yield_t_ha"]

        single.append({
            "start": sy,
            "end": ey,
            "land_pct": (a1 * y0 - a0 * y0).sum() / (a1 * y1 - a0 * y0).sum() * 100,
        })

single = pd.DataFrame(single)
print(single.round(1).to_string(index=False))
print(f"\nrange: {single.land_pct.min():.1f}% to {single.land_pct.max():.1f}%")

 start  end  land_pct
  1990 2021     171.2
  1990 2022     163.5
  1990 2023     165.0
  1991 2021     158.5
  1991 2022     150.5
  1991 2023     152.0
  1992 2021     176.7
  1992 2022     167.1
  1992 2023     168.9

range: 150.5% to 176.7%


A 26 point spread, just from which years I happen to pick. That is why I use
averages.

## 2. Why a simple before-and-after split does not work

Production is area times yield. When both change, part of the growth belongs to both of
them at once: the new land, farmed at the new yield.

A simple split has to give that shared part to one side or the other, so the answer
depends on which change I apply first. Both sums below are correct.

In [5]:
total_gain = (ends["prod_end"] - ends["prod_start"]).sum()

# order 1: land changes first, yield second
land_a  = (ends["area_ha_end"] * ends["yield_t_ha_start"] - ends["prod_start"]).sum()
yield_a = total_gain - land_a

# order 2: yield changes first, land second
yield_b = (ends["area_ha_start"] * (ends["yield_t_ha_end"] - ends["yield_t_ha_start"])).sum()
land_b  = total_gain - yield_b

shared = ((ends["area_ha_end"] - ends["area_ha_start"])
          * (ends["yield_t_ha_end"] - ends["yield_t_ha_start"])).sum()

print(f"land first:   land {land_a / total_gain * 100:6.1f}%   yield {yield_a / total_gain * 100:6.1f}%")
print(f"yield first:  land {land_b / total_gain * 100:6.1f}%   yield {yield_b / total_gain * 100:6.1f}%")
print()
print(f"total gain:  {total_gain:,.0f} tonnes")
print(f"shared part: {shared:,.0f} tonnes ({shared / total_gain * 100:.0f}% of the gain)")

land first:   land  163.8%   yield  -63.8%
yield first:  land  110.0%   yield  -10.0%

total gain:  96,667,370 tonnes
shared part: -51,933,871 tonnes (-54% of the gain)


The two answers are 54 points apart, because the shared part is 54% of the total
gain. I cannot pick one of them and defend it.

## 3. Using logs instead

A log turns a multiplication into an addition. An addition has no order, so the problem
goes away.

For each crop, the area log plus the yield log should equal the production log exactly.
The check below confirms it.

In [6]:
ends["ln_area"]  = np.log(ends["area_ha_end"] / ends["area_ha_start"])
ends["ln_yield"] = np.log(ends["yield_t_ha_end"] / ends["yield_t_ha_start"])
ends["ln_prod"]  = np.log(ends["prod_end"] / ends["prod_start"])
ends["left_over"] = ends["ln_area"] + ends["ln_yield"] - ends["ln_prod"]

if ends["left_over"].abs().max() > 1e-12:
    raise ValueError("the log split should be exact")

ends[["crop", "ln_area", "ln_yield", "ln_prod", "left_over"]].round(6)

,crop,ln_area,ln_yield,ln_prod,left_over
0,Cassava,1.468888,-0.581216,0.887672,-0.0
1,Cowpea,0.840827,0.253556,1.094383,0.0
2,Groundnut,1.298545,-0.115957,1.182588,0.0
3,Maize,0.135025,0.611330,0.746355,-0.0
4,Millet,-0.865541,-0.056904,-0.922445,0.0
5,Rice,1.086597,-0.037505,1.049091,0.0
6,Sorghum,0.131393,0.124053,0.255446,0.0
7,Yam,1.578416,-0.284229,1.294187,-0.0


Nothing left over, for any crop.

### Weighting the crops

Cassava is about 39% of this food basket and millet about 4%, so I cannot treat them
equally.

Each crop's share changed over the period, so I use the average of its share at the start
and at the end.

In [7]:
ends["share_start"] = ends["prod_start"] / ends["prod_start"].sum()
ends["share_end"]   = ends["prod_end"] / ends["prod_end"].sum()
ends["weight"]      = (ends["share_start"] + ends["share_end"]) / 2

if abs(ends["weight"].sum() - 1) > 1e-12:
    raise ValueError("weights should add up to 1")

ends[["crop", "share_start", "share_end", "weight"]].round(4).sort_values(
    "weight", ascending=False)

,crop,share_start,share_end,weight
0,Cassava,0.3968,0.3805,0.3887
7,Yam,0.2658,0.3827,0.3242
3,Maize,0.0921,0.0767,0.0844
6,Sorghum,0.0817,0.0416,0.0616
5,Rice,0.0477,0.0538,0.0507
4,Millet,0.0726,0.0114,0.0420
1,Cowpea,0.0225,0.0265,0.0245
2,Groundnut,0.0208,0.0268,0.0238


In [8]:
area_part  = (ends["weight"] * ends["ln_area"]).sum()
yield_part = (ends["weight"] * ends["ln_yield"]).sum()
both       = area_part + yield_part

share_area  = area_part / both * 100
share_yield = yield_part / both * 100
growth_pct  = (np.exp(both) - 1) * 100

print(f"production growth: {growth_pct:.1f}%")
print(f"  from more land:  {share_area:.1f}%")
print(f"  from yield:      {share_yield:.1f}%")

production growth: 149.1%
  from more land:  128.4%
  from yield:      -28.4%


**Nigeria's output of these eight crops grew 149%, and all of it came from planting
more land. Yield pulled in the other direction.**

In [9]:
ends["area_growth_pct"]  = (ends["area_ha_end"] / ends["area_ha_start"] - 1) * 100
ends["yield_growth_pct"] = (ends["yield_t_ha_end"] / ends["yield_t_ha_start"] - 1) * 100

ends[["crop", "area_growth_pct", "yield_growth_pct", "weight"]].round(1).sort_values(
    "area_growth_pct", ascending=False)

,crop,area_growth_pct,yield_growth_pct,weight
7,Yam,384.7,-24.7,0.3
0,Cassava,334.4,-44.1,0.4
2,Groundnut,266.4,-10.9,0.0
5,Rice,196.4,-3.7,0.1
1,Cowpea,131.8,28.9,0.0
3,Maize,14.5,84.3,0.1
6,Sorghum,14.0,13.2,0.1
4,Millet,-57.9,-5.5,0.0


Maize is the exception. Its yield rose 84% on almost the same land, which shows
yield growth is possible in Nigeria.

## 4. Comparing with other countries

Niger has the highest cassava yield in this group, but it only farms 23,000 hectares of
it against Nigeria's 9.9 million. Egypt has the highest cowpea yield on under 2,000
irrigated hectares against Nigeria's 5 million. Neither is a useful target.

So a country only counts as a peer for a crop if it farms at least 10% as much of that
crop as Nigeria does.

In [10]:
ref = panel[panel["year"].between(*C.REF_WINDOW)]
avg = ref.groupby(["country", "crop"])[["yield_t_ha", "area_ha"]].mean().reset_index()


def build_gap(min_share):
    rows = []
    for crop, g in avg.groupby("crop"):
        ng = g[g["country"] == "Nigeria"]
        if ng.empty:
            continue
        ng_yield = ng["yield_t_ha"].values[0]
        ng_area  = ng["area_ha"].values[0]

        peers = g[(g["country"] != "Nigeria") & (g["area_ha"] >= min_share * ng_area)]
        if peers.empty:
            continue
        best = peers.loc[peers["yield_t_ha"].idxmax()]

        rows.append({
            "crop": crop,
            "nigeria_t_ha": ng_yield,
            "peer_median_t_ha": peers["yield_t_ha"].median(),
            "best_t_ha": best["yield_t_ha"],
            "best_country": best["country"],
            "qualifying_peers": len(peers),
            "nigeria_area_ha": ng_area,
            "upside_t": ng_area * (best["yield_t_ha"] - ng_yield),
        })

    return pd.DataFrame(rows).sort_values("upside_t", ascending=False).reset_index(drop=True)


gap = build_gap(C.MIN_PEER_AREA_SHARE)
gap.round(2)

,crop,nigeria_t_ha,peer_median_t_ha,best_t_ha,best_country,qualifying_peers,nigeria_area_ha,upside_t
0,Cassava,5.95,6.84,23.70,Ghana,3,9908319.2,1.759192e+08
1,Maize,2.03,1.95,7.14,Egypt,11,6107560.8,3.121466e+07
2,Rice,1.95,3.16,9.11,Egypt,4,4346334.0,3.113745e+07
3,Sorghum,1.16,1.04,2.69,Ethiopia,6,5705500.0,8.701990e+06
4,Millet,1.00,0.98,2.50,Ethiopia,6,1855494.0,2.780708e+06
5,Groundnut,1.21,0.86,1.34,Senegal,7,3554788.2,4.718501e+05
6,Cowpea,0.84,0.48,0.49,Mali,3,4975144.2,-1.740163e+06
7,Yam,8.01,5.52,5.52,Côte d'Ivoire,1,7296948.6,-1.821089e+07


### Does the 10% cut-off change the answer?

In [11]:
for share in [0.0, 0.002, 0.01, 0.05, 0.10, 0.25, 0.50]:
    g = build_gap(share)
    cassava = g[g["crop"] == "Cassava"]
    best = cassava["best_country"].values[0] if len(cassava) else "none"
    print(f"{share:>6.1%}   crops compared: {len(g)}   cassava benchmark: {best}")

  0.0%   crops compared: 8   cassava benchmark: Niger
  0.2%   crops compared: 8   cassava benchmark: Niger
  1.0%   crops compared: 8   cassava benchmark: Ghana
  5.0%   crops compared: 8   cassava benchmark: Ghana
 10.0%   crops compared: 8   cassava benchmark: Ghana
 25.0%   crops compared: 6   cassava benchmark: none
 50.0%   crops compared: 4   cassava benchmark: none


Niger only becomes the cassava benchmark when the cut-off is near zero, because it
farms 0.2% of Nigeria's cassava area. Anything from 1% to 10% gives Ghana. Above 25% too
few countries qualify to compare at all.

So 10% is in the stable middle, not on an edge.

In [12]:
above_median  = int((gap["nigeria_t_ha"] > gap["peer_median_t_ha"]).sum())
upside        = gap.loc[gap["upside_t"] > 0, "upside_t"]
cassava_share = gap.loc[gap["crop"] == "Cassava", "upside_t"].values[0] / upside.sum() * 100
leads         = gap.loc[gap["upside_t"] < 0, "crop"].tolist()

print(f"above the peer median on {above_median} of {len(gap)} crops")
print(f"ahead of its best peer on: {', '.join(leads)}")
print(f"cassava is {cassava_share:.0f}% of the total upside")

above the peer median on 6 of 8 crops
ahead of its best peer on: Cowpea, Yam
cassava is 70% of the total upside


### What this says

Nigeria is not broadly behind. It beats the typical comparable country on most of these
crops and leads on yam.

The gap is almost all cassava, which is Nigeria's biggest crop by area and the one where
Ghana gets about four times the yield. That one crop is 70% of the upside.

That is more useful than saying Nigeria is behind, because it says where to put the
effort.

## 5. Save

In [13]:
ends.to_csv(C.PROCESSED / "decomposition.csv", index=False)
gap.to_csv(C.PROCESSED / "yield_gap.csv", index=False)

headline = {
    "window": {
        "start": C.START_YEAR,
        "end": C.END_YEAR,
        "endpoint_years_averaged": C.ENDPOINT_WINDOW,
    },
    "production_growth_pct": round(float(growth_pct), 1),
    "share_from_area_pct": round(float(share_area), 1),
    "share_from_yield_pct": round(float(share_yield), 1),
    "naive_split": {
        "land_applied_first_pct": round(float(land_a / total_gain * 100), 1),
        "yield_applied_first_pct": round(float(land_b / total_gain * 100), 1),
        "interaction_tonnes": int(round(shared)),
        "interaction_share_of_gain_pct": round(float(shared / total_gain * 100), 1),
    },
    "endpoint_sensitivity": {
        "single_year_min_pct": round(float(single.land_pct.min()), 1),
        "single_year_max_pct": round(float(single.land_pct.max()), 1),
    },
    "yield_gap": {
        "reference_window": list(C.REF_WINDOW),
        "min_peer_area_share": C.MIN_PEER_AREA_SHARE,
        "crops_compared": int(len(gap)),
        "crops_above_peer_median": above_median,
        "crops_leading_best_peer": leads,
        "cassava_share_of_upside_pct": round(float(cassava_share)),
        "total_upside_tonnes": int(round(upside.sum())),
    },
}

with open(C.PROCESSED / "headline.json", "w") as f:
    json.dump(headline, f, indent=2)

print(json.dumps(headline, indent=2))

{
  "window": {
    "start": 1990,
    "end": 2023,
    "endpoint_years_averaged": 3
  },
  "production_growth_pct": 149.1,
  "share_from_area_pct": 128.4,
  "share_from_yield_pct": -28.4,
  "naive_split": {
    "land_applied_first_pct": 163.8,
    "yield_applied_first_pct": 110.0,
    "interaction_tonnes": -51933871,
    "interaction_share_of_gain_pct": -53.7
  },
  "endpoint_sensitivity": {
    "single_year_min_pct": 150.5,
    "single_year_max_pct": 176.7
  },
  "yield_gap": {
    "reference_window": [
      2019,
      2023
    ],
    "min_peer_area_share": 0.1,
    "crops_compared": 8,
    "crops_above_peer_median": 6,
    "crops_leading_best_peer": [
      "Cowpea",
      "Yam"
    ],
    "cassava_share_of_upside_pct": 70,
    "total_upside_tonnes": 250225883
  }
}


I write the headline numbers to a file instead of typing them anywhere else. The
dashboard and the README read from it, so nothing can go out of date.

---

**Next:** `04_model.ipynb` checks whether fertilizer explains any of the gap.